In [ ]:
import torch
import math
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
words = open("data/names.txt", "r").read().splitlines()
words[:8]

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

In [ ]:
block_size = 3 # context length

def build_dataset(words, block_size=3):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [ ]:
g = torch.Generator()
C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)
W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [ ]:
sum(p.nelement() for p in parameters) # number of parameters in total

In [ ]:
for p in parameters:
  p.requires_grad = True

In [ ]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [ ]:
lri = []
lossi = []
stepi = []

In [ ]:
for i in range(200000):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (32,))
  
  # forward pass
  emb = C[Xtr[ix]] # (32, 3, 2)
  h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Ytr[ix])
  #print(loss.item())
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  #lr = lrs[i]
  lr = 0.1 if i < 100000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  #lri.append(lre[i])
  stepi.append(i)
  lossi.append(loss.log10().item())

#print(loss.item())

In [ ]:
plt.plot(stepi, lossi)

In [ ]:
# training loss 
emb = C[Xtr] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
# validation loss
emb = C[Xdev] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Ydev)
loss

In [ ]:
# test loss
emb = C[Xte] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 30) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Yte)
loss

In [ ]:
# visualize dimensions 0 and 1 of the embedding matrix C for all characters
plt.figure(figsize=(8,8))
plt.scatter(C[:,0].data, C[:,1].data, s=200)
for i in range(C.shape[0]):
    plt.text(C[i,0].item(), C[i,1].item(), itos[i], ha="center", va="center", color='white')
plt.grid('minor')

In [ ]:
## sampling from model
g = torch.Generator()

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

### Hyperparameter Optimisation

In [ ]:
def plot_e(losses_dict, title, xlabel='Step', window=100, val_steps=None):
    def smooth(values):
        return [sum(values[max(0, i-window):i+1]) / len(values[max(0, i-window):i+1]) for i in range(len(values))]
    
    plt.figure(figsize=(10, 4))
    for label, losses in losses_dict.items():
        if val_steps and label == 'val':
            plt.plot(val_steps, losses, label=label)
        else:
            plt.plot(smooth(losses), label=label)
    plt.legend()
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(f'Log10 loss (smoothed, window={window})')
    plt.show()

In [ ]:
### Hidden layer neurons 

n_counts = [50, 70, 100, 150]
n_losses = {}

for n in n_counts:
    g = torch.Generator()
    C = torch.randn((27, 10), generator=g)
    W1 = torch.randn((30, n), generator=g)
    b1 = torch.randn(n, generator=g)
    W2 = torch.randn((n, 27), generator=g)
    b2 = torch.randn(27, generator=g)
    params = [C,W1,b1,W2,b2]
    for p in params: 
        p.requires_grad = True

    losses = []
    for i in range(40000):
        ix = torch.randint(0,Xtr.shape[0], (32,))
        emb = C[Xtr[ix]]
        h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits,Ytr[ix])
        for p in params:
            p.grad = None
        loss.backward()
        for p in params:
            p.data += -0.1 * p.grad
        losses.append(loss.log10().item())
    n_losses[n] = losses

plot_e(n_losses, "hidden layer neuron count")

In [ ]:
### embedding dimentions
emb_dims = [15, 20, 30, 40]
emb_losses = {}

for d in emb_dims:
    g = torch.Generator()
    C = torch.randn((27, d), generator=g)
    W1 = torch.randn((block_size * d, 200), generator=g)
    b1 = torch.randn(200, generator=g)
    W2 = torch.randn((200, 27), generator=g)
    b2 = torch.randn(27, generator=g)
    params = [C, W1, b1, W2, b2]
    for p in params:
        p.requires_grad = True

    losses = []
    for i in range(40000):
        ix = torch.randint(0, Xtr.shape[0], (32,))
        emb = C[Xtr[ix]]
        h = torch.tanh(emb.view(-1, block_size * d) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits, Ytr[ix])
        for p in params:
            p.grad = None
        loss.backward()
        for p in params:
            p.data += -0.1 * p.grad
        losses.append(loss.log10().item())
    emb_losses[d] = losses

plot_e(emb_losses, "embedding dimentionality")

In [ ]:
block_sizes = [2, 3, 4, 6]
ctx_losses = {}

for bs in block_sizes:
    Xtr_bs, Ytr_bs = build_dataset(words[:n1], bs)

    g = torch.Generator()
    C = torch.randn((27, 10), generator=g)
    W1 = torch.randn((bs * 10, 200), generator=g)
    b1 = torch.randn(200, generator=g)
    W2 = torch.randn((200, 27), generator=g)
    b2 = torch.randn(27, generator=g)
    params = [C, W1, b1, W2, b2]
    for p in params:
        p.requires_grad = True

    losses = []
    for i in range(30000):
        ix = torch.randint(0, Xtr_bs.shape[0], (32,))
        emb = C[Xtr_bs[ix]]
        h = torch.tanh(emb.view(-1, bs * 10) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits, Ytr_bs[ix])
        for p in params:
            p.grad = None
        loss.backward()
        for p in params:
            p.data += -0.1 * p.grad
        losses.append(loss.log10().item())
    ctx_losses[bs] = losses

plot_e(ctx_losses, "context length")

In [ ]:
schedules = {
    'constant_0.1':  lambda i: 0.1,
    'constant_0.01': lambda i: 0.01,
    'step_decay':    lambda i: 0.1 if i < 15000 else 0.01,
    'cosine':        lambda i: 0.01 + 0.5 * (0.1 - 0.01) * (1 + torch.cos(torch.tensor(i / 30000 * 3.14159)).item()),
}

sched_losses = {}

for name, lr_fn in schedules.items():
    g = torch.Generator()
    C = torch.randn((27, 10), generator=g)
    W1 = torch.randn((30, 200), generator=g)
    b1 = torch.randn(200, generator=g)
    W2 = torch.randn((200, 27), generator=g)
    b2 = torch.randn(27, generator=g)
    params = [C, W1, b1, W2, b2]
    for p in params:
        p.requires_grad = True

    losses = []
    for i in range(15000):
        ix = torch.randint(0, Xtr.shape[0], (32,))
        emb = C[Xtr[ix]]
        h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
        logits = h @ W2 + b2
        loss = F.cross_entropy(logits, Ytr[ix])
        for p in params:
            p.grad = None
        loss.backward()
        lr = lr_fn(i)
        for p in params:
            p.data += -lr * p.grad
        losses.append(loss.log10().item())
    sched_losses[name] = losses

plot_e(sched_losses, "optimisation scheudle")

### Tuned Hyperparams

In [ ]:
# val loss to beat = 2.163

# hidden layer neurons = 300
# emb dimentionality = 40
# ctx length = 3
# optimisation schedule = cosine

## define params
n_count = 200 
emb_dim = 30
ctx_length = 3
lr_fn = lambda i: 0.01 + 0.5 * (0.1 - 0.01) * (1 + torch.cos(torch.tensor(i / 30000 * 3.14159)).item())

In [ ]:
g = torch.Generator()
C = torch.randn((27, emb_dim), generator=g)
W1 = torch.randn((ctx_length * emb_dim, n_count), generator=g)
b1 = torch.randn(n_count, generator=g)
W2 = torch.randn((n_count, 27), generator=g)
b2 = torch.randn(27, generator=g)
params = [C,W1,b1,W2,b2]

In [ ]:
for p in params:
  p.requires_grad = True

lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre
lri = []
lossi = []
stepi = []

In [ ]:
losses = []
val_losses = []
val_steps = []

for i in range(200000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, ctx_length * emb_dim) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    for p in params:
        p.grad = None
    loss.backward()
    # lr = lr_fn(i)
    lr = 0.1 if i < 100000 else 0.01
    
    for p in params:
        p.data += -lr * p.grad
    if i % 1000 == 0:
        with torch.no_grad():
            emb = C[Xdev]
            h = torch.tanh(emb.view(-1, ctx_length * emb_dim) @ W1 + b1)
            logits = h @ W2 + b2
            val_loss = F.cross_entropy(logits, Ydev).item()
        val_losses.append(math.log10(val_loss))
        val_steps.append(i)
    losses.append(loss.log10().item())

plot_e({'train': losses, 'val': val_losses}, 'Train vs val loss', val_steps=val_steps)

In [ ]:
# training loss 
emb = C[Xtr]
h = torch.tanh(emb.view(-1, ctx_length * emb_dim) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)
loss

In [ ]:
# validation loss
emb = C[Xdev]
h = torch.tanh(emb.view(-1, ctx_length * emb_dim) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
loss

In [ ]:
# test loss
emb = C[Xte]
h = torch.tanh(emb.view(-1, ctx_length * emb_dim) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Yte)
loss

### Sampling

In [ ]:
g = torch.Generator().manual_seed(122 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size
    while True:
      emb = C[torch.tensor([context])]
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))